# UNet training — Greenhouse dataset

Uses the dataset prepared by `dataset_to_unetdataset.py`:

```
unet_dataset/
├── train/images, train/masks
├── val/images,   val/masks
└── test/images,  test/masks
```

Masks are binary (0 = background, 255 = foreground).
Outputs go to `OUT_DIR`: best/last checkpoints, `history.csv` (per-epoch train/val metrics),
`test_results.csv`, `summary.csv` (best model on train/val/test), and the charts.

In [ ]:
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ==== CONFIG ====
DATA_ROOT = Path("./unet_dataset")      # output of dataset_to_unetdataset.py
OUT_DIR   = Path("./unet_results")

IMG_SIZE    = 256      # images/masks are resized to IMG_SIZE x IMG_SIZE
BATCH_SIZE  = 16
EPOCHS      = 30
LR          = 1e-3
NUM_WORKERS = 4        # set to 0 if you get DataLoader errors on Windows
BASE_CH     = 32       # UNet width (32 -> ~7.8M params, 64 -> ~31M)
THRESHOLD   = 0.5      # probability threshold for foreground
PATIENCE    = 7        # early stopping on val Dice (None to disable)
SEED        = 42

# Optional: limit samples per split for a quick test run (None = use all)
MAX_SAMPLES = {"train": None, "val": None, "test": None}
# ================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

## Dataset

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def find_pairs(split_dir: Path):
    """Match images and masks by filename stem."""
    imgs  = {p.stem: p for p in (split_dir / "images").iterdir() if p.suffix.lower() in IMG_EXTS}
    masks = {p.stem: p for p in (split_dir / "masks").iterdir()  if p.suffix.lower() in IMG_EXTS}
    stems = sorted(imgs.keys() & masks.keys())
    if len(stems) != len(imgs) or len(stems) != len(masks):
        print(f"  WARNING {split_dir.name}: {len(imgs)} images, {len(masks)} masks, {len(stems)} paired")
    return [(imgs[s], masks[s]) for s in stems]


class SegDataset(Dataset):
    def __init__(self, pairs, img_size, augment=False):
        self.pairs = pairs
        self.size = (img_size, img_size)
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        img_p, mask_p = self.pairs[i]
        img  = Image.open(img_p).convert("RGB").resize(self.size, Image.BILINEAR)
        mask = Image.open(mask_p).convert("L").resize(self.size, Image.NEAREST)

        img  = np.asarray(img, dtype=np.float32) / 255.0
        mask = (np.asarray(mask) > 127).astype(np.float32)   # 0/255 -> 0/1

        if self.augment:
            if np.random.rand() < 0.5:                        # horizontal flip
                img, mask = img[:, ::-1], mask[:, ::-1]
            if np.random.rand() < 0.5:                        # vertical flip
                img, mask = img[::-1], mask[::-1]

        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        img  = torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)))
        mask = torch.from_numpy(np.ascontiguousarray(mask))[None]   # (1, H, W)
        return img, mask


def _worker_init(_):
    # Each worker is a fork of the notebook kernel, so it inherits copies of old
    # DataLoader iterators that the kernel hasn't garbage-collected yet. When the
    # worker collects them, their __del__ tries to stop workers it doesn't own and
    # prints "AssertionError: can only test a child process". Workers never own
    # iterators themselves, so disabling that cleanup inside the worker is safe.
    from torch.utils.data import dataloader as _dl
    _dl._MultiProcessingDataLoaderIter.__del__ = lambda self: None


def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                      worker_init_fn=_worker_init if NUM_WORKERS > 0 else None,
                      persistent_workers=False)


datasets, loaders = {}, {}
for split in ["train", "val", "test"]:
    pairs = find_pairs(DATA_ROOT / split)
    if MAX_SAMPLES.get(split):
        pairs = pairs[:MAX_SAMPLES[split]]
    datasets[split] = SegDataset(pairs, IMG_SIZE, augment=(split == "train"))
    loaders[split] = make_loader(datasets[split], shuffle=(split == "train"))
    print(f"{split:>5}: {len(datasets[split])} samples")

## Model

In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )


class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        ch = [base, base * 2, base * 4, base * 8, base * 16]
        self.downs = nn.ModuleList()
        prev = in_ch
        for c in ch:
            self.downs.append(DoubleConv(prev, c))
            prev = c
        self.ups, self.dec = nn.ModuleList(), nn.ModuleList()
        for c in reversed(ch[:-1]):
            self.ups.append(nn.ConvTranspose2d(prev, c, 2, stride=2))
            self.dec.append(DoubleConv(c * 2, c))
            prev = c
        self.head = nn.Conv2d(prev, out_ch, 1)

    def forward(self, x):
        skips = []
        for i, d in enumerate(self.downs):
            x = d(x)
            if i < len(self.downs) - 1:
                skips.append(x)
                x = F.max_pool2d(x, 2)
        for up, dec, s in zip(self.ups, self.dec, reversed(skips)):
            x = up(x)
            if x.shape[-2:] != s.shape[-2:]:
                x = F.interpolate(x, size=s.shape[-2:], mode="bilinear", align_corners=False)
            x = dec(torch.cat([s, x], dim=1))
        return self.head(x)


model = UNet(base=BASE_CH).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

## Loss & metrics

In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.smooth = smooth

    def forward(self, logits, target):
        bce = self.bce(logits, target)
        p = torch.sigmoid(logits)
        inter = (p * target).sum(dim=(1, 2, 3))
        union = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        dice = 1 - (2 * inter + self.smooth) / (union + self.smooth)
        return bce + dice.mean()


class MetricTracker:
    """Accumulates pixel counts over a whole epoch (dataset-level metrics)."""
    def __init__(self):
        self.tp = self.fp = self.fn = self.tn = 0
        self.loss_sum = 0.0
        self.n = 0

    @torch.no_grad()
    def update(self, logits, target, loss):
        pred = torch.sigmoid(logits) > THRESHOLD
        t = target > 0.5
        self.tp += (pred & t).sum().item()
        self.fp += (pred & ~t).sum().item()
        self.fn += (~pred & t).sum().item()
        self.tn += (~pred & ~t).sum().item()
        self.loss_sum += loss.item() * target.size(0)
        self.n += target.size(0)

    def compute(self):
        eps = 1e-7
        tp, fp, fn, tn = self.tp, self.fp, self.fn, self.tn
        return {
            "loss":      self.loss_sum / max(self.n, 1),
            "dice":      2 * tp / (2 * tp + fp + fn + eps),
            "iou":       tp / (tp + fp + fn + eps),
            "precision": tp / (tp + fp + eps),
            "recall":    tp / (tp + fn + eps),
            "pixel_acc": (tp + tn) / (tp + fp + fn + tn + eps),
        }

## Train / evaluate functions

In [ ]:
criterion = BCEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def run_epoch(loader, train: bool, desc: str):
    gc.collect()   # free finished loader iterators before new workers are forked
    model.train(train)
    tracker = MetricTracker()
    with torch.set_grad_enabled(train):
        for x, y in tqdm(loader, desc=desc, leave=False):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            tracker.update(logits.float(), y, loss)
    return tracker.compute()

## Training (train + val)
Metrics are written to `history.csv` after every epoch, so nothing is lost if training stops early.

In [ ]:
history = []
best_dice, best_epoch, bad_epochs = -1.0, 0, 0
best_path = OUT_DIR / "best_model.pt"
history_csv = OUT_DIR / "history.csv"

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr = run_epoch(loaders["train"], train=True,  desc=f"Epoch {epoch} train")
    va = run_epoch(loaders["val"],   train=False, desc=f"Epoch {epoch} val")
    scheduler.step(va["dice"])

    row = {"epoch": epoch, "lr": optimizer.param_groups[0]["lr"], "time_s": round(time.time() - t0, 1)}
    row.update({f"train_{k}": v for k, v in tr.items()})
    row.update({f"val_{k}": v for k, v in va.items()})
    history.append(row)
    pd.DataFrame(history).to_csv(history_csv, index=False)

    improved = va["dice"] > best_dice
    if improved:
        best_dice, best_epoch, bad_epochs = va["dice"], epoch, 0
        torch.save(model.state_dict(), best_path)
    else:
        bad_epochs += 1

    print(f"Epoch {epoch:3d} | "
          f"train loss {tr['loss']:.4f} dice {tr['dice']:.4f} iou {tr['iou']:.4f} | "
          f"val loss {va['loss']:.4f} dice {va['dice']:.4f} iou {va['iou']:.4f} | "
          f"{row['time_s']}s" + ("  *best*" if improved else ""))

    if PATIENCE and bad_epochs >= PATIENCE:
        print(f"Early stopping: no val Dice improvement for {PATIENCE} epochs.")
        break

torch.save(model.state_dict(), OUT_DIR / "last_model.pt")
print(f"\nBest val Dice {best_dice:.4f} at epoch {best_epoch}. History -> {history_csv}")

## Test evaluation (best checkpoint)
The best model (by val Dice) is evaluated on train, val and test. Train is evaluated without augmentation so the three numbers are comparable.

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))

final = {}
for split in ["train", "val", "test"]:
    datasets[split].augment = False
    eval_loader = make_loader(datasets[split], shuffle=False)
    final[split] = run_epoch(eval_loader, train=False, desc=f"Final {split}")
datasets["train"].augment = True

test_df = pd.DataFrame([{"split": "test", "best_epoch": best_epoch, **final["test"]}])
test_df.to_csv(OUT_DIR / "test_results.csv", index=False)

summary_df = pd.DataFrame([{"split": s, "best_epoch": best_epoch, **m} for s, m in final.items()])
summary_df.to_csv(OUT_DIR / "summary.csv", index=False)

print("Best model evaluated on all splits:")
display(summary_df.round(4))

## Charts

In [ ]:
hist = pd.read_csv(history_csv)
plot_metrics = ["loss", "dice", "iou", "precision", "recall", "pixel_acc"]
colors = {"train": "tab:blue", "val": "tab:orange", "test": "tab:green"}

# 1) per-epoch curves: train & val lines, test result as a dashed line
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, m in zip(axes.flat, plot_metrics):
    ax.plot(hist["epoch"], hist[f"train_{m}"], marker="o", ms=3, label="train", color=colors["train"])
    ax.plot(hist["epoch"], hist[f"val_{m}"],   marker="o", ms=3, label="val",   color=colors["val"])
    ax.axhline(final["test"][m], ls="--", color=colors["test"],
               label=f"test (best ep {best_epoch}) = {final['test'][m]:.4f}")
    ax.axvline(best_epoch, ls=":", color="gray", alpha=0.6)
    ax.set_title(m)
    ax.set_xlabel("epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("UNet — train / val curves with test result")
fig.tight_layout()
fig.savefig(OUT_DIR / "curves_train_val_test.png", dpi=150)
plt.show()

# 2) bar chart: best model on train / val / test
bar_metrics = ["dice", "iou", "precision", "recall", "pixel_acc"]
x = np.arange(len(bar_metrics))
w = 0.27
fig, ax = plt.subplots(figsize=(11, 5))
for i, split in enumerate(["train", "val", "test"]):
    vals = [final[split][m] for m in bar_metrics]
    bars = ax.bar(x + (i - 1) * w, vals, w, label=split, color=colors[split])
    ax.bar_label(bars, fmt="%.3f", fontsize=7, padding=2)
ax.set_xticks(x, bar_metrics)
ax.set_ylim(0, 1.08)
ax.set_title(f"Best model (epoch {best_epoch}) — train / val / test")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "bars_train_val_test.png", dpi=150)
plt.show()

## Sample test predictions

In [ ]:
N_SHOW = 4
model.eval()
rng = np.random.default_rng(SEED)
idx = rng.choice(len(datasets["test"]), size=min(N_SHOW, len(datasets["test"])), replace=False)

fig, axes = plt.subplots(len(idx), 3, figsize=(10, 3.3 * len(idx)), squeeze=False)
with torch.no_grad():
    for row, i in enumerate(idx):
        x, y = datasets["test"][i]
        pred = (torch.sigmoid(model(x[None].to(DEVICE)))[0, 0] > THRESHOLD).cpu().numpy()
        img = (x.numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)
        for ax, im, title in zip(axes[row], [img, y[0].numpy(), pred], ["image", "ground truth", "prediction"]):
            ax.imshow(im, cmap=None if title == "image" else "gray")
            ax.set_title(title)
            ax.axis("off")
fig.tight_layout()
fig.savefig(OUT_DIR / "test_samples.png", dpi=120)
plt.show()